# Lab 8.8 &mdash; Critique & Debate

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Run a propose -> critique -> revise loop
- Let an independent critic gate on quality
- Cap the loop so two agents can't argue forever

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Critique & debate](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-08"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Instead of averaging, **stress-test** one answer (deck slide 13): one agent **proposes**, an independent
**critic** tries to find what's wrong, the proposer **revises** &mdash; repeat until the critic approves or a
**cap** is hit. Generating and evaluating are **different skills**, so a separate skeptic catches the
author's blind spots (just like code review). Always **cap** the loop.

In [ ]:
# proposer improves its draft each round; critic approves only a GROUNDED answer.
def proposer(prev):
    if prev is None: return "draft v1"
    if "v1" in prev: return "draft v2 grounded"
    return prev
def critic(answer):
    return "approve" if "grounded" in answer else "revise"
def critic_never(answer):
    return "revise"
print("proposer & critics ready")

## Build it
Complete `critique_loop`: get the critic's verdict and stop when it approves (or at the cap).

In [ ]:
def critique_loop(proposer, critic, max_rounds=3):
    answer = None
    for r in range(max_rounds):
        answer = proposer(answer)
        verdict = ___   # TODO: ask the critic to review this answer
        if ___:   # TODO: the critic approved
            return {"answer": answer, "rounds": r + 1, "approved": True}
    return {"answer": answer, "rounds": max_rounds, "approved": False}

try:
    print("approved:", critique_loop(proposer, critic))
    print("capped  :", critique_loop(proposer, critic_never))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The loop approves in **round 2** &mdash; the proposer's revision earns the critic's OK, and the result carries the `"grounded"` answer.
- A **never-satisfied** critic hits the **cap** and returns `approved: False` &mdash; the loop can argue, but not forever.
- The critic **gates** the outcome: quality comes from the separate skeptic, not from the proposer marking its own work.

## Your turn (open task &mdash; no grader)
Write a stricter `critic` that also requires the word *"cited"*, and a `proposer` that only adds it on round 3.
**What good looks like:** approval now takes an extra round (and a higher `max_rounds` to reach it), proving the
critic &mdash; not the proposer &mdash; sets the quality bar. Drop the cap too low and a good answer never
lands: tune the cap to the task.

---
### Key takeaway
A separate critic raises quality because evaluating is a different skill from generating -- and the cap keeps the debate from running forever. Use it when being right beats being fast.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>